# Building tools

Assistant version 4. Read [the module guide](README.md) for objectives, diagrams, costs and exercises.

Run all cells in order. These cells replay synthetic fixtures and block external traffic. They do not measure live model quality. Each example's code is editable below; its original command remains in the docstring. Use the CLI without `--mock` only after reviewing provider setup and cost.


In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists() and (p / "data").is_dir()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
from common.network import offline_network


def run_lesson(entry_point, script):
    """Replay the script offline, on a fresh worker event loop when needed."""

    def work():
        previous = sys.argv
        sys.argv = [script, "--mock"]
        try:
            with offline_network():
                entry_point()
        finally:
            sys.argv = previous

    with ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(work).result(timeout=120)

## 01_typed_search

Source: [01_typed_search.py](examples/01_typed_search.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""@tool; docstrings, parameter descriptions and type hints becoming tool specifications.

Run: uv run python modules/05_building_tools/examples/01_typed_search.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No. Resources created: none.
"""

from strands import Agent

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli
from common.testing import tool_message
from common.tools.booking import search_flights


def main() -> None:
    _, cfg = lesson_settings()
    console.print(search_flights.tool_spec)
    # KEY IDEA: type hints and Args descriptions become the model-visible input schema.
    schema = search_flights.tool_spec["inputSchema"]["json"]
    assert "destination" in schema["required"]
    model = get_model(
        cfg,
        responses=[
            tool_message("search_flights", {"destination": "Lisbon"}),
            "I found synthetic flights; confirm a date before choosing.",
        ],
    )
    agent = Agent(model=model, tools=[search_flights], callback_handler=None, retry_strategy=None)
    complete(agent("Search Nimbus flights to Lisbon.", limits={"turns": 3}), cfg)


run_lesson(main, "modules/05_building_tools/examples/01_typed_search.py")

## 02_results_and_errors

Source: [02_results_and_errors.py](examples/02_results_and_errors.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""toolUse, toolResult, status and tool exception feedback.

Run: uv run python modules/05_building_tools/examples/02_results_and_errors.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No. Resources created: none.
"""

from strands import Agent, tool

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli
from common.testing import tool_message


@tool
def validate_reference(reference: str) -> dict:
    """Validate the Nimbus booking prefix before lookup.

    Args:
        reference: Booking reference supplied by the traveler.
    """
    if not reference.startswith("BKG-"):
        return {"status": "error", "content": [{"text": "Reference must start BKG-"}]}
    raise ValueError("Booking service unavailable in this failure demo")


def main() -> None:
    _, cfg = lesson_settings()
    model = get_model(
        cfg,
        responses=[
            tool_message("validate_reference", {"reference": "bad"}),
            tool_message("validate_reference", {"reference": "BKG-0001"}, "retry"),
            "Please retry later; no booking was changed.",
        ],
    )
    agent = Agent(
        model=model, tools=[validate_reference], callback_handler=None, retry_strategy=None
    )
    result = agent("Validate bad, then BKG-0001; explain any failure.", limits={"turns": 4})
    # KEY IDEA: both explicit errors and exceptions become toolResult feedback.
    statuses = [
        b["toolResult"]["status"] for m in agent.messages for b in m["content"] if "toolResult" in b
    ]
    console.print("Tool statuses:", statuses)
    if cfg.provider == "mock":
        assert statuses == ["error", "error"]
    complete(result, cfg)


run_lesson(main, "modules/05_building_tools/examples/02_results_and_errors.py")

## 03_tool_context

Source: [03_tool_context.py](examples/03_tool_context.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""@tool(context=True), ToolContext, private agent state and invocation state.

Run: uv run python modules/05_building_tools/examples/03_tool_context.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No. Resources created: none.
"""

from strands import Agent, ToolContext, tool

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import run_cli
from common.testing import tool_message


@tool(context=True)
def traveler_preferences(tool_context: ToolContext) -> dict:
    """Read trusted identity and private seat preference for this invocation."""
    return {
        "customer": tool_context.invocation_state["customer_id"],
        "seat": tool_context.agent.state.get("seat"),
    }


def main() -> None:
    _, cfg = lesson_settings()
    model = get_model(
        cfg,
        responses=[
            tool_message("traveler_preferences", {}),
            "Traveler CUS-001 prefers an aisle seat.",
        ],
    )
    agent = Agent(
        model=model,
        tools=[traveler_preferences],
        state={"seat": "aisle"},
        callback_handler=None,
        retry_strategy=None,
    )
    # KEY IDEA: context is injected by the application, not chosen as a model argument.
    assert "tool_context" not in traveler_preferences.tool_spec["inputSchema"]["json"]["properties"]
    result = agent(
        "Read my saved seat preference.",
        invocation_state={"customer_id": "CUS-001"},
        limits={"turns": 3},
    )
    complete(result, cfg)


run_lesson(main, "modules/05_building_tools/examples/03_tool_context.py")

## 04_async_progress

Source: [04_async_progress.py](examples/04_async_progress.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Async tools, streamed tool progress and consumption.

Run: uv run python modules/05_building_tools/examples/04_async_progress.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No. Resources created: none.
"""

import asyncio

from strands import Agent, tool

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli
from common.testing import tool_message


@tool
async def check_trip():
    """Stream progress while checking synthetic trip availability."""
    yield {"progress": "checking flights"}
    await asyncio.sleep(0)
    yield {"progress": "checking hotels"}
    # The final yielded item becomes the tool's returned value.
    yield {"status": "success", "content": [{"text": "Synthetic inventory checked."}]}


async def run(cfg):
    agent = Agent(
        model=get_model(cfg, responses=[tool_message("check_trip", {}), "Trip checks finished."]),
        tools=[check_trip],
        callback_handler=None,
        retry_strategy=None,
    )
    result = None
    async for event in agent.stream_async("Check my trip.", limits={"turns": 3}):
        # KEY IDEA: progress arrives before the final agent answer.
        if "tool_stream_event" in event:
            console.print("Progress:", event["tool_stream_event"]["data"])
        result = event.get("result", result)
    return result


def main() -> None:
    _, cfg = lesson_settings()
    complete(asyncio.run(run(cfg)), cfg)


run_lesson(main, "modules/05_building_tools/examples/04_async_progress.py")

## 05_resource_sharing

Source: [05_resource_sharing.py](examples/05_resource_sharing.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Class-based tools sharing a database connection; read-only queries.

Run: uv run python modules/05_building_tools/examples/05_resource_sharing.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No. Resources created: none.
"""

import sqlite3
import threading
from contextlib import closing

from strands import Agent, tool

from common.data import data_path
from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli


class BookingDesk:
    def __init__(self, connection):
        self.connection = connection
        self.lock = threading.Lock()

    @tool
    def booking_total(self, booking_id: str) -> int:
        """Read a synthetic booking total in cents.

        Args:
            booking_id: Booking ID in the demo database; use only synthetic fixtures.
        """
        with self.lock:
            row = self.connection.execute(
                "SELECT total_cents FROM bookings WHERE booking_id=?", (booking_id,)
            ).fetchone()
        if row is None:
            raise ValueError("Unknown booking")
        return row[0]


def main() -> None:
    _, cfg = lesson_settings()
    # KEY IDEA: SDK calls run on another thread; serialize access to this shared read-only handle.
    with closing(
        sqlite3.connect(
            data_path("bookings.db").as_uri() + "?mode=ro", uri=True, check_same_thread=False
        )
    ) as connection:
        desk = BookingDesk(connection)
        agent = Agent(
            model=get_model(cfg, responses=[] if cfg.provider == "mock" else None),
            tools=[desk.booking_total],
            callback_handler=None,
        )
        result = agent.tool.booking_total(booking_id="BKG-0001")
        console.print(result)
        assert result["status"] == "success"
    console.print(
        "For model-driven concurrent tools, open one connection per call as booking.py does."
    )
    complete()


run_lesson(main, "modules/05_building_tools/examples/05_resource_sharing.py")

## 06_module_tools

Source: [06_module_tools.py](examples/06_module_tools.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""TOOL_SPEC modules and direct agent.tool calls.

Run: uv run python modules/05_building_tools/examples/06_module_tools.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No. Resources created: none.
"""

from strands import Agent

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli
from common.tools import policy_lookup


def main() -> None:
    _, cfg = lesson_settings()
    agent = Agent(
        model=get_model(cfg, responses=[] if cfg.provider == "mock" else None),
        tools=[policy_lookup],
        callback_handler=None,
    )
    # KEY IDEA: direct calls use registered tool schemas but do not ask the model to choose.
    result = agent.tool.policy_lookup()
    assert result["status"] == "success"
    console.print(result)
    console.print("Direct calls recorded in history:", len(agent.messages))
    complete()


run_lesson(main, "modules/05_building_tools/examples/06_module_tools.py")

## 07_tool_executors

Source: [07_tool_executors.py](examples/07_tool_executors.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Sequential and concurrent execution; race and resource limits.

Run: uv run python modules/05_building_tools/examples/07_tool_executors.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No. Resources created: none.
"""

import asyncio
import time

from strands import Agent, tool
from strands.tools.executors import ConcurrentToolExecutor, SequentialToolExecutor

from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli
from common.testing import tool_message


@tool
async def inventory_check(section: str) -> str:
    """Read one synthetic inventory section without mutating shared state.

    Args:
        section: Either flights or hotels.
    """
    await asyncio.sleep(0.02)
    return section + " available"


def main() -> None:
    _, cfg = lesson_settings()
    calls = {
        "role": "assistant",
        "content": [
            tool_message("inventory_check", {"section": name}, name)["content"][0]
            for name in ("flights", "hotels")
        ],
    }
    for executor in (SequentialToolExecutor(), ConcurrentToolExecutor()):
        agent = Agent(
            model=get_model(cfg, responses=[calls, "Both inventories checked."]),
            tools=[inventory_check],
            tool_executor=executor,
            callback_handler=None,
            retry_strategy=None,
        )
        start = time.perf_counter()
        # KEY IDEA: independent reads can overlap; shared mutations need coordination.
        result = agent("Check flights and hotels together.", limits={"turns": 3})
        console.print(type(executor).__name__, "elapsed:", round(time.perf_counter() - start, 3))
        complete(result, cfg)


run_lesson(main, "modules/05_building_tools/examples/07_tool_executors.py")

## 08_tool_directory

Source: [08_tool_directory.py](examples/08_tool_directory.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Trusted folder loading, hot reload and review before registration.

Run: uv run python modules/05_building_tools/examples/08_tool_directory.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: No. Resources created: none.
"""

import contextlib

from strands import Agent

from common.config import ROOT
from common.lessons import complete, lesson_settings
from common.models import get_model
from common.pretty import console, run_cli


def main() -> None:
    _, cfg = lesson_settings()
    folder = ROOT / ".nimbus/tool_reload"
    (folder / "tools").mkdir(parents=True, exist_ok=True)
    tool_path = folder / "tools/nimbus_status.py"
    original = (
        "from strands import tool\n@tool\ndef nimbus_status() -> str:\n"
        '    "Read reviewed Nimbus status."\n'
        "    return 'ready'\n"
    )
    tool_path.write_text(original, encoding="utf-8")
    # KEY IDEA: importing a tool file executes Python. Only load operator-reviewed code.
    with contextlib.chdir(folder):
        agent = Agent(
            model=get_model(cfg, responses=[] if cfg.provider == "mock" else None),
            load_tools_from_directory=True,
            callback_handler=None,
        )
        console.print(agent.tool.nimbus_status())
        tool_path.write_text(
            original.replace("'ready'", "'updated and reviewed'"), encoding="utf-8"
        )
        # Explicit reload makes the walkthrough deterministic; the watcher also reloads on edits.
        agent.tool_registry.reload_tool("nimbus_status")
        result = agent.tool.nimbus_status()
        assert "updated and reviewed" in str(result)
        console.print(result)
        agent.cleanup()
    complete()


run_lesson(main, "modules/05_building_tools/examples/08_tool_directory.py")

## 09_manual_converse_loop

Source: [09_manual_converse_loop.py](examples/09_manual_converse_loop.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Build it yourself: bounded boto3 Converse loop with toolConfig/toolUse/toolResult, then
Strands equivalent.

Run: uv run python modules/05_building_tools/examples/09_manual_converse_loop.py --mock
Expected: Lesson complete.
Cost: mock/local $0 hosted inference; live hosted calls use tokens.
AWS required: Yes; offline Stubber replay. Resources created: none.
"""

from contextlib import nullcontext

import boto3
from botocore.config import Config
from botocore.stub import Stubber
from strands import Agent

from common.config import load_settings
from common.lessons import complete, lesson_settings, require_bedrock
from common.models import get_model
from common.pretty import console, run_cli
from common.testing import tool_message
from common.tools import policy_lookup


def manual_loop(client, model_id: str) -> dict:
    messages = [{"role": "user", "content": [{"text": "Read Nimbus refund policy."}]}]
    specs = {"tools": [{"toolSpec": policy_lookup.TOOL_SPEC}]}
    for _ in range(3):
        response = client.converse(
            modelId=model_id,
            messages=messages,
            toolConfig=specs,
            inferenceConfig={"maxTokens": 256, "temperature": 0},
        )
        message = response["output"]["message"]
        messages.append(message)
        if response["stopReason"] == "end_turn":
            return message
        results = []
        for block in message["content"]:
            if "toolUse" in block:
                call = block["toolUse"]
                if call["name"] != "policy_lookup":
                    raise ValueError("Tool not allowed")
                results.append({"toolResult": policy_lookup.policy_lookup(call)})
        if not results:
            raise ValueError("No answer or allowed tool request")
        # KEY IDEA: append toolResult as a user message, then ask the model again.
        messages.append({"role": "user", "content": results})
    raise RuntimeError("Manual loop exceeded three turns")


def main() -> None:
    args, cfg = lesson_settings()
    target = load_settings(provider="bedrock")
    if args.mock:
        client = boto3.client(
            "bedrock-runtime",
            region_name=target.aws_region,
            aws_access_key_id="testing",
            aws_secret_access_key="testing",
        )
        stub = Stubber(client)
        for message, stop in [
            (tool_message("policy_lookup", {}), "tool_use"),
            ({"role": "assistant", "content": [{"text": "24 hours and 7 days."}]}, "end_turn"),
        ]:
            stub.add_response(
                "converse",
                {
                    "output": {"message": message},
                    "stopReason": stop,
                    "usage": {"inputTokens": 100, "outputTokens": 20, "totalTokens": 120},
                    "metrics": {"latencyMs": 0},
                },
            )
        context = stub
    else:
        require_bedrock(cfg)
        client = boto3.Session(
            profile_name=cfg.aws_profile or None, region_name=cfg.aws_region
        ).client(
            "bedrock-runtime",
            config=Config(connect_timeout=5, read_timeout=30, retries={"max_attempts": 1}),
        )
        context = nullcontext()
    with context:
        console.print("Manual:", manual_loop(client, target.model_id))
    model = get_model(cfg, responses=[tool_message("policy_lookup", {}), "24 hours and 7 days."])
    agent = Agent(model=model, tools=[policy_lookup], callback_handler=None, retry_strategy=None)
    result = agent("Read Nimbus refund policy.", limits={"turns": 3})
    complete(result, cfg)


run_lesson(main, "modules/05_building_tools/examples/09_manual_converse_loop.py")

## booking_assistant

Source: [booking_assistant.py](use_case/booking_assistant.py). Predict the output, run the cell, then inspect the messages and metrics.


In [ ]:
"""Search flights/hotels, check seats, and inspect an authorized booking.

Run: uv run python modules/05_building_tools/use_case/booking_assistant.py --mock
Expected: a Nimbus result followed by Lesson complete.
Cost: mock/local $0 hosted inference; hosted mode uses model tokens.
AWS resources created: none. Concepts: compose this module's taught APIs.
"""

from common.assistant import booking_agent
from common.lessons import complete, lesson_settings
from common.pretty import run_cli
from common.testing import tool_message
from common.tools.booking import search_flights


def main() -> None:
    _, cfg = lesson_settings()
    first = search_flights("Lisbon")[0]
    replies = [
        tool_message("search_flights", {"destination": "Lisbon"}),
        tool_message("search_hotels", {"city": "Lisbon"}, "hotels"),
        tool_message("check_seat_availability", {"flight_id": first["flight_id"]}, "seats"),
        tool_message("get_booking", {"booking_id": "BKG-0001"}, "booking"),
        "Synthetic options found; booking BKG-0001 is confirmed. No reservation was made.",
    ]
    agent = booking_agent(cfg, replies)
    # KEY IDEA: the authenticated application supplies identity, outside model arguments.
    result = agent(
        "Search Lisbon flights and hotels, check one flight's seats, then my BKG-0001 booking.",
        invocation_state={"customer_id": "CUS-001"},
        limits={"turns": 6},
    )
    complete(result, cfg)


run_lesson(main, "modules/05_building_tools/use_case/booking_assistant.py")

## Practice next

Complete the three exercises in the [module guide](README.md#exercises), then answer the [quiz](quiz.md). Run the same acceptance tests against your edited starters.
